# IN403 – Unit 10 Lab: Production Monitoring, Alerting, and Incident Response

## Objective
You will analyze production telemetry for two hospital AI systems and create an **alert + response plan**.

### Systems
- **No-Show Predictor** (tabular model)
- **Policy RAG Assistant** (LLM + retrieval)

## Deliverables
1) Identify which days should trigger alerts and why
2) Create an alert dashboard table (your thresholds)
3) Draft a mini runbook response for one critical alert
4) Recommend a remediation plan (retrain, rollback, safe-mode, re-index docs, etc.)

## Files provided
- `IN403_Unit10_Production_Monitoring_Dataset.zip`
  - `production_metrics_daily.csv`
  - `user_feedback_log.csv`
  - `production_runbook.txt`


In [ ]:
# Optional installs (Colab)
# !pip -q install pandas numpy matplotlib

import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Load dataset


In [ ]:
zip_file = 'IN403_Unit10_Production_Monitoring_Dataset.zip'
data_dir = 'unit10_data'

if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(data_dir)

nmetrics = pd.read_csv(os.path.join(data_dir, 'production_metrics_daily.csv'))
feedback = pd.read_csv(os.path.join(data_dir, 'user_feedback_log.csv'))
runbook = open(os.path.join(data_dir, 'production_runbook.txt'), 'r', encoding='utf-8').read()

display(metrics.head())
display(feedback.head())
print(runbook[:600])


## 2) Separate metrics by system and visualize trends
Create at least 2 charts per system.


In [ ]:
ns = metrics[metrics['system_name']=='No-Show Predictor'].copy()
rag = metrics[metrics['system_name']=='Policy RAG Assistant'].copy()

ns['date'] = pd.to_datetime(ns['date'])
rag['date'] = pd.to_datetime(rag['date'])

plt.figure(figsize=(10,4))
plt.plot(ns['date'], ns['metric_auc'])
plt.title('No-Show Predictor: AUC Over Time')
plt.ylabel('AUC'); plt.xlabel('date'); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
plt.plot(ns['date'], ns['metric_drift_max_psi'])
plt.title('No-Show Predictor: Drift (Max PSI) Over Time')
plt.ylabel('max PSI'); plt.xlabel('date'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(rag['date'], rag['metric_retrieval_hit_rate'])
plt.title('Policy RAG: Retrieval Hit Rate Over Time')
plt.ylabel('hit rate'); plt.xlabel('date'); plttight_layout(); plt.show()

plt.figure(figsize=(10,4))
plt.plot(rag['date'], rag['metric_hallucination_rate'])
plt.title('Policy RAG: Hallucination Rate Over Time')
plt.ylabel('hallucination rate'); plt.xlabel('date'); plt.tight_layout(); plt.show()


## 3) Define your alert thresholds

You must define thresholds for:
- No-show: drift (PSI), fairness gap, AUC/performance, errors
- RAG: retrieval hit rate, hallucination rate, latency

Then flag all days that violate your thresholds.


In [ ]:
# Example thresholds (edit + justify)
TH = {
  'ns_auc_min': 0.80,
  'ns_psi_max': 0.25,
  'ns_fair_max': 0.12,
  'ns_err_max': 8,
  'rag_hit_min': 0.70,
  'rag_hallu_max': 0.08,
  'rag_lat_max': 1500
}

s_alert = (
    (ns['metric_auc'] < TH['ns_auc_min']) |
    (ns['metric_drift_max_psi'] > TH['ns_psi_max']) |
    (ns['metric_fairness_tpr_gap'] > TH['ns_fair_max']) |
    (ns['error_count'] > TH['ns_err_max'])
)

rag_alert = (
    (rag['metric_retrieval_hit_rate'] < TH['rag_hit_min']) |
    (rag['metric_hallucination_rate'] > TH['rag_hallu_max']) |
    (rag['metric_latency_p95_ms'] > TH['rag_lat_max'])
)

ns['my_alert'] = ns_alert.astype(int)
rag['my_alert'] = rag_alert.astype(int)

display(ns[ns['my_alert']==1][['date','metric_auc','metric_drift_max_psi','metric_fairness_tpr_gap','error_count','my_alert']].head())
display(rag[rag['my_alert']==1][['date','metric_retrieval_hit_rate','metric_hallucination_rate','metric_latency_p95_ms','my_alert']].head())


## 4) Connect feedback to alerts

Aggregate feedback by week and compare against alerts.
Answer: do user complaints rise when telemetry degrades?


In [ ]:
feedback['date'] = pd.to_datetime(feedback['date'])
feedback['week'] = feedback['date'].dt.to_period('W').astype(str)
agg = (feedback.groupby(['system_name','week'])
       .agg(avg_rating=('rating_1to5','mean'),
            issue_count=('reported_issue', lambda s: (s!='none').sum()))
       .reset_index())
display(agg.head())


## 5) Incident response mini-runbook

Pick ONE day with an alert and write a response plan.
Your plan must include:
- What fired + severity
- Containment action (safe-mode, HITL, rollback)
- Diagnostics you would run
- Remediation steps
- Verification and re-approval triggers


### Student runbook response (write here)
- **Alert day:** 
- **System:** 
- **What fired:** 
- **Containment:** 
- **Diagnostics:** 
- **Remediation:** 
- **Verification plan:** 
- **Re-approval triggers:** 
